# Agente 1 del TFM: lenguaje natural → `yfinance` → CSV

Este notebook implementa una versión **corregida y más robusta** del primer agente de tu arquitectura.

## Qué corrige respecto al notebook anterior

1. **Ya no confunde nombres de empresa con tickers.**  
   Antes, `"Nvidia"` se estaba tratando como ticker directo `"NVIDIA"`, y eso llevaba a errores en la descarga.

2. **El parser entiende mejor español natural.**  
   Ahora reconoce mejor expresiones como:
   - `Cuánto ha crecido Nvidia en 5 años`
   - `Descárgame el histórico del S&P 500 desde 2020`
   - `Quiero el oro en 1 semana a 1h`
   - `Compara Nvidia y AMD en 2 años`
   - `Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31`
   - `Muéstrame la evolución de Apple en 5 años`

3. **La resolución de activos es híbrida.**  
   Primero prueba alias locales; si no basta, usa búsqueda dinámica de `yfinance` (`Lookup` y `Search`).

4. **Incluye un pequeño caché local** de resoluciones activo → ticker.

5. **Intenta mitigar errores SSL** típicos de algunos entornos Windows configurando `certifi`.

6. **Devuelve un CSV** con el `raw` descargado por `yf.download(...)`, y además un `.metadata.json`.

> **Idea del agente**:  
> mensaje del usuario → parser → resolución de ticker → construcción de parámetros → `yf.download(...)` → CSV


In [12]:
%pip install -q yfinance pandas python-dateutil certifi

Note: you may need to restart the kernel to use updated packages.


## 1. Imports y configuración base

En esta celda:
- importamos librerías,
- configuramos certificados SSL con `certifi` para reducir errores de entorno,
- y dejamos preparada la carpeta de exportación.


In [13]:
import os
import re
import json
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Tuple
from datetime import datetime

import pandas as pd
from dateutil.relativedelta import relativedelta
import yfinance as yf

# Ayuda en algunos entornos Windows / corporativos donde yfinance
# puede fallar por certificados SSL.
try:
    import certifi
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
    os.environ.setdefault("CURL_CA_BUNDLE", certifi.where())
except Exception:
    pass

EXPORT_DIR = Path("exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Alias locales y utilidades generales

Aquí dejamos un conjunto **pequeño y mantenible** de alias frecuentes.
No intentamos meter un diccionario gigantesco: eso sería difícil de mantener.

La estrategia correcta para el agente es:
- alias locales para los casos más comunes,
- y resolución dinámica para el resto.


In [14]:
VALID_INTERVALS = {
    "1m", "2m", "5m", "15m", "30m", "60m", "90m",
    "1h", "1d", "5d", "1wk", "1mo", "3mo"
}

INTRADAY_INTERVALS = {"1m", "2m", "5m", "15m", "30m", "60m", "90m", "1h"}


def utc_now_naive() -> pd.Timestamp:
    return pd.Timestamp.now(tz="UTC").tz_localize(None)


def today_floor() -> pd.Timestamp:
    return utc_now_naive().normalize()


def normalize_text(text: str) -> str:
    text = text.strip()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))


ALIASES_RAW = {
    # Acciones frecuentes
    "nvidia": "NVDA",
    "amd": "AMD",
    "apple": "AAPL",
    "microsoft": "MSFT",
    "tesla": "TSLA",
    "amazon": "AMZN",
    "google": "GOOG",
    "alphabet": "GOOG",
    "meta": "META",
    "netflix": "NFLX",

    # Índices
    "sp500": "^GSPC",
    "s&p 500": "^GSPC",
    "s&p500": "^GSPC",
    "s p 500": "^GSPC",
    "sandp 500": "^GSPC",
    "nasdaq": "^IXIC",
    "nasdaq 100": "^NDX",
    "dow jones": "^DJI",
    "ibex 35": "^IBEX",
    "euro stoxx 50": "^STOXX50E",
    "dax": "^GDAXI",
    "cac 40": "^FCHI",
    "nikkei 225": "^N225",

    # Cripto
    "bitcoin": "BTC-USD",
    "btc": "BTC-USD",
    "ethereum": "ETH-USD",
    "eth": "ETH-USD",

    # Materias primas / futuros
    "oro": "GC=F",
    "gold": "GC=F",
    "plata": "SI=F",
    "silver": "SI=F",
    "petroleo": "CL=F",
    "petróleo": "CL=F",
    "crudo": "CL=F",
    "brent": "BZ=F",
    "gas natural": "NG=F",
    "cobre": "HG=F",

    # Divisas
    "eurusd": "EURUSD=X",
    "eur/usd": "EURUSD=X",
    "usd/jpy": "JPY=X",
    "gbp/usd": "GBPUSD=X",
}

ALIASES = {normalize_text(k): v for k, v in ALIASES_RAW.items()}

STOPWORD_PATTERNS = [
    r"\b(muestrame|muéstrame|ensename|enséñame|dame|sacame|sácame|quiero|necesito|descargame|descárgame|descarga|obten|obt[eé]n)\b",
    r"\b(cuanto|cuánto)\s+(ha|a)\s+(crecido|evolucionado|subido|bajado)\b",
    r"\b(evolucion|evolución)\s+de\b",
    r"\b(historico|histórico)\s+de(l)?\b",
    r"\b(datos|dato)\s+de\b",
    r"\b(precio|precios|cotizacion|cotización)\s+de\b",
    r"\b(compara|comparar|comparame|compárame)\b",
    r"\b(accion|acciones|índice|indice)\s+de\b",
]

UNIT_MAP = {
    "ano": "years", "anos": "years", "año": "years", "años": "years",
    "year": "years", "years": "years",
    "mes": "months", "meses": "months",
    "month": "months", "months": "months",
    "semana": "weeks", "semanas": "weeks",
    "week": "weeks", "weeks": "weeks",
    "dia": "days", "dias": "days", "día": "days", "días": "days",
    "day": "days", "days": "days",
    "hora": "hours", "horas": "hours",
    "hour": "hours", "hours": "hours",
}

UNIT_RE = (
    r"(ano|anos|año|años|mes|meses|semana|semanas|dia|dias|día|días|"
    r"hora|horas|year|years|month|months|week|weeks|day|days|hour|hours)"
)


def looks_like_direct_ticker(text: str) -> bool:
    """Heurística conservadora para no confundir 'Nvidia' con ticker directo."""
    raw = text.strip()
    if not raw or " " in raw:
        return False
    if any(ch in raw for ch in "^=-."):
        return bool(re.fullmatch(r"[\^A-Za-z0-9][A-Za-z0-9=\-\.]{0,14}", raw))
    return raw.isupper() and bool(re.fullmatch(r"[A-Z]{1,5}", raw))


def to_yyyy_mm_dd(ts: pd.Timestamp) -> str:
    return pd.Timestamp(ts).strftime("%Y-%m-%d")


def inclusive_to_exclusive_end(end_date: str) -> str:
    return (pd.Timestamp(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")


def slugify(text: str) -> str:
    text = strip_accents(text.lower())
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")[:80] or "consulta"


def load_cache(cache_path: str = "ticker_cache.json") -> Dict[str, Any]:
    cache_file = Path(cache_path)
    if not cache_file.exists():
        return {}
    try:
        return json.loads(cache_file.read_text(encoding="utf-8"))
    except Exception:
        return {}


def save_cache(cache: Dict[str, Any], cache_path: str = "ticker_cache.json") -> None:
    Path(cache_path).write_text(
        json.dumps(cache, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


## 3. Parser de lenguaje natural

Este bloque:
- detecta el intervalo,
- detecta el rango temporal,
- limpia el texto,
- extrae uno o varios activos.

La corrección clave aquí es que **ya no se normaliza el activo demasiado pronto**.
En el notebook anterior eso ayudó a confundir `Nvidia` con un ticker directo.


In [15]:
def apply_relative_range(qty: int, unit_text: str) -> Tuple[str, str]:
    unit_key = UNIT_MAP[normalize_text(unit_text)]
    now = today_floor()

    if unit_key == "years":
        start = now - relativedelta(years=qty)
    elif unit_key == "months":
        start = now - relativedelta(months=qty)
    elif unit_key == "weeks":
        start = now - relativedelta(weeks=qty)
    elif unit_key == "days":
        start = now - relativedelta(days=qty)
    elif unit_key == "hours":
        start = (utc_now_naive() - relativedelta(hours=qty)).floor("min")
    else:
        start = now - relativedelta(years=1)

    end = now + pd.Timedelta(days=1)
    return to_yyyy_mm_dd(start), to_yyyy_mm_dd(end)


def extract_interval(text: str) -> Tuple[str, str]:
    interval = "1d"
    patterns = [
        r"(?:a|cada|intervalo(?: de)?)\s*(1m|2m|5m|15m|30m|60m|90m|1h|1d|5d|1wk|1mo|3mo)\b",
        r"\b(1m|2m|5m|15m|30m|60m|90m|1h|1d|5d|1wk|1mo|3mo)\b",
    ]
    cleaned = text

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            interval = match.group(1)
            cleaned = re.sub(pattern, " ", cleaned, flags=re.IGNORECASE)
            break

    return interval, re.sub(r"\s+", " ", cleaned).strip()


def extract_date_range(text: str) -> Tuple[str, str, str, List[str]]:
    notes: List[str] = []
    cleaned = text

    between = re.search(
        r"desde\s+(\d{4}-\d{2}-\d{2}|\d{4})\s+hasta\s+(\d{4}-\d{2}-\d{2}|\d{4})",
        text,
        flags=re.IGNORECASE,
    )
    if between:
        raw_start, raw_end = between.group(1), between.group(2)
        start = f"{raw_start}-01-01" if re.fullmatch(r"\d{4}", raw_start) else raw_start
        end_inclusive = f"{raw_end}-12-31" if re.fullmatch(r"\d{4}", raw_end) else raw_end
        cleaned = re.sub(re.escape(between.group(0)), " ", cleaned, flags=re.IGNORECASE)
        return start, inclusive_to_exclusive_end(end_inclusive), re.sub(r"\s+", " ", cleaned).strip(), notes

    since = re.search(r"desde\s+(\d{4}-\d{2}-\d{2}|\d{4})", text, flags=re.IGNORECASE)
    if since:
        raw_start = since.group(1)
        start = f"{raw_start}-01-01" if re.fullmatch(r"\d{4}", raw_start) else raw_start
        end = to_yyyy_mm_dd(today_floor() + pd.Timedelta(days=1))
        cleaned = re.sub(re.escape(since.group(0)), " ", cleaned, flags=re.IGNORECASE)
        return start, end, re.sub(r"\s+", " ", cleaned).strip(), notes

    numeric_rel = re.search(
        rf"(?:en|ultimos?|últimos?|ultimas?|últimas?|de)\s+(\d+)\s+{UNIT_RE}\b",
        text,
        flags=re.IGNORECASE,
    )
    if numeric_rel:
        qty = int(numeric_rel.group(1))
        unit = numeric_rel.group(2)
        start, end = apply_relative_range(qty, unit)
        cleaned = re.sub(re.escape(numeric_rel.group(0)), " ", cleaned, flags=re.IGNORECASE)
        return start, end, re.sub(r"\s+", " ", cleaned).strip(), notes

    single_rel = re.search(
        rf"(?:en|ultimo|último|ultima|última)\s+(un|una)\s+{UNIT_RE}\b",
        text,
        flags=re.IGNORECASE,
    )
    if single_rel:
        unit = single_rel.group(2)
        start, end = apply_relative_range(1, unit)
        cleaned = re.sub(re.escape(single_rel.group(0)), " ", cleaned, flags=re.IGNORECASE)
        return start, end, re.sub(r"\s+", " ", cleaned).strip(), notes

    last_rel = re.search(
        rf"(?:ultimo|último|ultima|última)\s+{UNIT_RE}\b",
        text,
        flags=re.IGNORECASE,
    )
    if last_rel:
        unit = last_rel.group(1)
        start, end = apply_relative_range(1, unit)
        cleaned = re.sub(re.escape(last_rel.group(0)), " ", cleaned, flags=re.IGNORECASE)
        return start, end, re.sub(r"\s+", " ", cleaned).strip(), notes

    if re.search(r"\bytd\b|este ano|este año", text, flags=re.IGNORECASE):
        now = today_floor()
        start = pd.Timestamp(year=now.year, month=1, day=1)
        end = now + pd.Timedelta(days=1)
        cleaned = re.sub(r"\bytd\b|este ano|este año", " ", cleaned, flags=re.IGNORECASE)
        return to_yyyy_mm_dd(start), to_yyyy_mm_dd(end), re.sub(r"\s+", " ", cleaned).strip(), notes

    now = today_floor()
    start = now - relativedelta(years=1)
    end = now + pd.Timedelta(days=1)
    notes.append("No se detectó rango temporal; se usa por defecto el último año.")
    return to_yyyy_mm_dd(start), to_yyyy_mm_dd(end), re.sub(r"\s+", " ", cleaned).strip(), notes


def clean_asset_chunk(text: str) -> str:
    cleaned = strip_accents(text)
    for pattern in STOPWORD_PATTERNS:
        cleaned = re.sub(pattern, " ", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\b(del|de|la|el|los|las|un|una)\b", " ", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s+", " ", cleaned).strip(" ,;:.")
    return cleaned


def extract_assets(text: str) -> List[str]:
    cleaned = clean_asset_chunk(text)
    if not cleaned:
        return []
    parts = re.split(r"\s*(?:,| y | e | vs | versus | contra | con )\s*", cleaned, flags=re.IGNORECASE)
    assets = [part.strip(" ,;:.") for part in parts if part.strip(" ,;:.")]
    return assets


def adjust_interval_for_range(start: str, end: str, interval: str) -> Tuple[str, List[str]]:
    notes: List[str] = []
    delta_days = (pd.Timestamp(end) - pd.Timestamp(start)).days

    if interval in INTRADAY_INTERVALS and delta_days > 60:
        notes.append(
            f"El intervalo solicitado ({interval}) es intradía y el rango supera 60 días. Se cambia automáticamente a 1d para respetar la limitación de Yahoo Finance."
        )
        return "1d", notes

    return interval, notes


def parse_user_request(user_text: str) -> Dict[str, Any]:
    interval, text_wo_interval = extract_interval(user_text)
    start, end, text_wo_dates, notes = extract_date_range(text_wo_interval)
    assets = extract_assets(text_wo_dates)

    if not assets:
        raise ValueError("No he podido detectar el activo solicitado en el mensaje.")

    interval, interval_notes = adjust_interval_for_range(start, end, interval)
    notes.extend(interval_notes)

    return {
        "original_query": user_text,
        "assets": assets,
        "start": start,
        "end": end,
        "interval": interval,
        "notes": notes,
    }


### Prueba rápida del parser

In [16]:
examples = [
    "Cuánto ha crecido nvidia en 5 años",
    "Descárgame el histórico del sp500 desde 2020",
    "Quiero el oro en 1 semana a 1h",
    "Compara nvidia y AMD en 2 años",
    "Datos de bitcoin desde 2024-01-01 hasta 2024-12-31",
    "Muéstrame la evolución de apple en 5 años",
]

for ex in examples:
    print("=" * 90)
    print(ex)
    print(json.dumps(parse_user_request(ex), ensure_ascii=False, indent=2))


Cuánto ha crecido nvidia en 5 años
{
  "original_query": "Cuánto ha crecido nvidia en 5 años",
  "assets": [
    "nvidia"
  ],
  "start": "2021-03-12",
  "end": "2026-03-13",
  "interval": "1d",
  "notes": []
}
Descárgame el histórico del sp500 desde 2020
{
  "original_query": "Descárgame el histórico del sp500 desde 2020",
  "assets": [
    "sp500"
  ],
  "start": "2020-01-01",
  "end": "2026-03-13",
  "interval": "1d",
  "notes": []
}
Quiero el oro en 1 semana a 1h
{
  "original_query": "Quiero el oro en 1 semana a 1h",
  "assets": [
    "oro"
  ],
  "start": "2026-03-05",
  "end": "2026-03-13",
  "interval": "1h",
  "notes": []
}
Compara nvidia y AMD en 2 años
{
  "original_query": "Compara nvidia y AMD en 2 años",
  "assets": [
    "nvidia",
    "AMD"
  ],
  "start": "2024-03-12",
  "end": "2026-03-13",
  "interval": "1d",
  "notes": []
}
Datos de bitcoin desde 2024-01-01 hasta 2024-12-31
{
  "original_query": "Datos de bitcoin desde 2024-01-01 hasta 2024-12-31",
  "assets": [
    

## 4. Resolución híbrida activo → ticker

La estrategia es:

1. Si el usuario ya escribió un ticker válido (`NVDA`, `BTC-USD`, `^GSPC`, `GC=F`), usarlo.
2. Si no, intentar con **alias locales**.
3. Si no, usar búsqueda dinámica de `yfinance`:
   - `Lookup` cuando está disponible,
   - y `Search` como refuerzo/fallback.
4. Guardar el resultado en caché.


In [17]:
def score_candidate(query: str, candidate: Dict[str, Any]) -> float:
    q = normalize_text(query)
    symbol = normalize_text(str(candidate.get("symbol", "")))
    shortname = normalize_text(str(candidate.get("shortname", "")))
    longname = normalize_text(str(candidate.get("longname", "")))
    quote_type = normalize_text(str(candidate.get("quoteType", "")))
    exchange = normalize_text(str(candidate.get("exchange", "")))

    score = 0.0
    if symbol == q:
        score += 100
    if q == shortname:
        score += 60
    if q == longname:
        score += 60
    if q in shortname:
        score += 30
    if q in longname:
        score += 30
    if q.replace(" ", "") == symbol.replace(" ", ""):
        score += 40
    if quote_type in {"equity", "etf", "index", "cryptocurrency", "currency", "future", "mutualfund"}:
        score += 10
    if exchange in {"nms", "nyq", "nas", "nyse", "nasdaq", "ccc", "cme"}:
        score += 5
    return score


def dedupe_candidates(candidates: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    deduped: List[Dict[str, Any]] = []
    seen = set()
    for item in candidates:
        symbol = str(item.get("symbol", "")).strip().upper()
        if not symbol or symbol in seen:
            continue
        seen.add(symbol)
        deduped.append(item)
    return deduped


def search_yfinance_candidates(query: str, max_results: int = 10) -> List[Dict[str, Any]]:
    candidates: List[Dict[str, Any]] = []

    # Lookup primero para ampliar cobertura por tipo de activo.
    if hasattr(yf, "Lookup"):
        for kwargs in (
            {"query": query, "raise_errors": False},
            {"query": query},
        ):
            try:
                lookup = yf.Lookup(**kwargs)
                for attr in ("stock", "etf", "index", "future", "currency", "cryptocurrency", "mutualfund"):
                    value = getattr(lookup, attr, None) or []
                    if isinstance(value, list):
                        candidates.extend(value[:max_results])
                if candidates:
                    break
            except TypeError:
                continue
            except Exception:
                pass

    # Search como refuerzo o fallback.
    search_kwargs_options = [
        {
            "query": query,
            "max_results": max_results,
            "news_count": 0,
            "lists_count": 0,
            "enable_fuzzy_query": True,
            "raise_errors": False,
        },
        {
            "query": query,
            "max_results": max_results,
            "news_count": 0,
            "enable_fuzzy_query": True,
        },
        {
            "query": query,
            "max_results": max_results,
        },
    ]

    for kwargs in search_kwargs_options:
        try:
            search = yf.Search(**kwargs)
            quotes = getattr(search, "quotes", []) or []
            if isinstance(quotes, list):
                candidates.extend(quotes)
            break
        except TypeError:
            continue
        except Exception:
            continue

    return dedupe_candidates(candidates)


def resolve_asset_to_ticker(asset: str, cache_path: str = "ticker_cache.json") -> Dict[str, Any]:
    cache = load_cache(cache_path)
    key = normalize_text(asset)

    if key in cache:
        return {
            "query": asset,
            "ticker": cache[key]["ticker"],
            "source": "cache",
            "match": cache[key],
        }

    if looks_like_direct_ticker(asset):
        ticker = asset.strip().upper()
        result = {
            "query": asset,
            "ticker": ticker,
            "source": "direct_ticker",
            "match": {"symbol": ticker},
        }
        cache[key] = {"ticker": result["ticker"], "source": result["source"]}
        save_cache(cache, cache_path)
        return result

    if key in ALIASES:
        result = {
            "query": asset,
            "ticker": ALIASES[key],
            "source": "alias",
            "match": {"symbol": ALIASES[key]},
        }
        cache[key] = {"ticker": result["ticker"], "source": result["source"]}
        save_cache(cache, cache_path)
        return result

    candidates = search_yfinance_candidates(asset)
    if not candidates:
        raise ValueError(f"No se pudo resolver el activo '{asset}' en Yahoo Finance / yfinance.")

    ranked = sorted(candidates, key=lambda c: score_candidate(asset, c), reverse=True)
    best = ranked[0]
    ticker = best.get("symbol")

    if not ticker:
        raise ValueError(f"No se encontró ticker válido para '{asset}'.")

    result = {
        "query": asset,
        "ticker": str(ticker).strip(),
        "source": "yfinance_lookup_search",
        "match": best,
        "alternatives": ranked[:5],
    }

    cache[key] = {"ticker": result["ticker"], "source": result["source"]}
    save_cache(cache, cache_path)
    return result


## 5. Construcción del plan de descarga y exportación a CSV

Aquí montamos exactamente la llamada estilo:

```python
raw = yf.download(
    tickers=TICKERS,
    start=START,
    end=END,
    interval=INTERVAL,
    group_by="ticker",
    auto_adjust=False,
    threads=True,
    progress=False
)
```

Además:
- guardamos el CSV,
- guardamos metadata útil,
- y devolvemos el `raw` para inspección.


In [18]:
def build_download_params(
    parsed: Dict[str, Any],
    resolved_assets: List[Dict[str, Any]],
    auto_adjust: bool = False,
) -> Dict[str, Any]:
    return {
        "tickers": [item["ticker"] for item in resolved_assets],
        "start": parsed["start"],
        "end": parsed["end"],
        "interval": parsed["interval"],
        "group_by": "ticker",
        "auto_adjust": auto_adjust,
        "threads": True,
        "progress": False,
    }


def explain_runtime_error(exc: Exception) -> str:
    msg = str(exc)
    lowered = msg.lower()
    if "ssl certificate" in lowered or "certificate verify failed" in lowered or "curl: (60)" in lowered:
        return (
            "Se ha detectado un problema de certificados SSL en tu entorno Python. "
            "He intentado mitigarlo con certifi, pero si persiste, actualiza certifi "
            "y revisa la configuración SSL de tu entorno o proxy corporativo.\n\n"
            f"Detalle original: {msg}"
        )
    return msg


def run_user_request(
    user_text: str,
    export_dir: str = "exports",
    auto_adjust: bool = False,
    cache_path: str = "ticker_cache.json",
) -> Dict[str, Any]:
    parsed = parse_user_request(user_text)

    try:
        resolved_assets = [
            resolve_asset_to_ticker(asset, cache_path=cache_path)
            for asset in parsed["assets"]
        ]
    except Exception as exc:
        raise RuntimeError(explain_runtime_error(exc)) from exc

    params = build_download_params(parsed, resolved_assets, auto_adjust=auto_adjust)

    try:
        raw = yf.download(
            tickers=params["tickers"],
            start=params["start"],
            end=params["end"],
            interval=params["interval"],
            group_by=params["group_by"],
            auto_adjust=params["auto_adjust"],
            threads=params["threads"],
            progress=params["progress"],
        )
    except Exception as exc:
        raise RuntimeError(explain_runtime_error(exc)) from exc

    if raw is None or raw.empty:
        raise ValueError("La descarga no devolvió datos. Revisa el ticker resuelto, el rango temporal o el intervalo.")

    export_path = Path(export_dir)
    export_path.mkdir(parents=True, exist_ok=True)

    filename_base = slugify(user_text) + "_" + datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = export_path / f"{filename_base}.csv"
    metadata_path = export_path / f"{filename_base}.metadata.json"

    raw.to_csv(csv_path)

    metadata = {
        "user_text": user_text,
        "parsed_request": parsed,
        "resolved_assets": resolved_assets,
        "download_params": params,
        "csv_path": str(csv_path),
        "row_count": int(len(raw)),
        "column_count": int(len(raw.columns)),
        "generated_at": datetime.now().isoformat(timespec="seconds"),
    }

    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )

    return {
        "raw": raw,
        "csv_path": str(csv_path),
        "metadata_path": str(metadata_path),
        "metadata": metadata,
    }


## 6. Uso real: una petición del usuario

Cambia `user_text` por la petición que quieras.  
Esta celda intentará:

1. interpretar el mensaje,
2. resolver los tickers,
3. descargar los datos,
4. y exportarlos a CSV.

> El archivo CSV generado contendrá el `raw` de `yf.download(...)`.


In [19]:
user_text = "Muéstrame la evolución de Apple en 5 años"

result = run_user_request(
    user_text=user_text,
    export_dir="exports",
    auto_adjust=False,
)

print("CSV generado:", result["csv_path"])
print("Metadata:", result["metadata_path"])
print()
print(json.dumps(result["metadata"], ensure_ascii=False, indent=2))
print()
result["raw"].head()


Failed to get ticker 'AAPL' reason: Failed to perform, curl: (60) SSL certificate problem: unable to get local issuer certificate. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$AAPL: possibly delisted; no timezone found

1 Failed download:
['AAPL']: possibly delisted; no timezone found


ValueError: La descarga no devolvió datos. Revisa el ticker resuelto, el rango temporal o el intervalo.

## 7. Procesar varias peticiones seguidas

Útil para probar varios ejemplos del TFM.


In [20]:
requests_batch = [
    "Cuánto ha crecido Nvidia en 5 años",
    "Descárgame el histórico del S&P 500 desde 2020",
    "Quiero el oro en 1 semana a 1h",
    "Compara Nvidia y AMD en 2 años",
    "Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31",
]

batch_results = []

for req in requests_batch:
    try:
        output = run_user_request(req, export_dir="exports", auto_adjust=False)
        batch_results.append({
            "query": req,
            "csv_path": output["csv_path"],
            "tickers": output["metadata"]["download_params"]["tickers"],
            "rows": output["metadata"]["row_count"],
        })
    except Exception as e:
        batch_results.append({
            "query": req,
            "error": str(e),
        })

pd.DataFrame(batch_results)


Failed to get ticker 'NVIDIA' reason: Failed to perform, curl: (60) SSL certificate problem: unable to get local issuer certificate. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$NVIDIA: possibly delisted; no timezone found

1 Failed download:
['NVIDIA']: possibly delisted; no timezone found
Failed to get ticker '^GSPC' reason: Failed to perform, curl: (60) SSL certificate problem: unable to get local issuer certificate. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$^GSPC: possibly delisted; no timezone found

1 Failed download:
['^GSPC']: possibly delisted; no timezone found
Failed to get ticker 'ORO' reason: Failed to perform, curl: (60) SSL certificate problem: unable to get local issuer certificate. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$ORO: possibly delisted; no timezone found

1 Failed download:
['ORO']: possibly delisted; no timezone found
Failed to get ticker 'AMD' reason: Failed to p

,query,error
0,Cuánto ha crecido Nvidia en 5 años,La descarga no devolvió datos. Revisa el ticke...
1,Descárgame el histórico del S&P 500 desde 2020,La descarga no devolvió datos. Revisa el ticke...
2,Quiero el oro en 1 semana a 1h,La descarga no devolvió datos. Revisa el ticke...
3,Compara Nvidia y AMD en 2 años,La descarga no devolvió datos. Revisa el ticke...
4,Datos de Bitcoin desde 2024-01-01 hasta 2024-1...,La descarga no devolvió datos. Revisa el ticke...


## 8. Notas y troubleshooting

### Si sigue apareciendo un error SSL
El problema ya no es del parser, sino del entorno Python. Prueba:
1. actualizar `certifi`,
2. reiniciar el kernel,
3. ejecutar de nuevo la celda de imports,
4. revisar si estás detrás de proxy o inspección SSL corporativa.

### Si una consulta no resuelve bien el ticker
Añade el alias al diccionario `ALIASES_RAW`.  
Esa es la combinación buena para el TFM:
- alias locales pequeños para los casos frecuentes,
- búsqueda dinámica para el resto.

### Qué genera este notebook
Por cada petición exitosa:
- un `.csv` con el `raw` descargado,
- y un `.metadata.json` con trazabilidad.
